# 05 · Limpieza: Base de datos afiliados ACTHO_anonimizada

Afiliados principales de ACTHO. Una fila = una empresa afiliada con su afiliado principal.

- Entrada: `datos/originales/Base de datos afiliados ACTHO_anonimizada.xlsx` (hoja `Hoja1`)
- Salida: `datos/limpios/05_afiliados_actho_limpio.csv`

Es una base pequeña y casi limpia (4 columnas, 472 filas). El trabajo es dejar los mismos nombres de columna que las demás bases
(`id_empresa`, `id_persona`, `ciudad`, `nivel_cargo`) para poder cruzarlas en Looker Studio.

In [ ]:
from pathlib import Path

import pandas as pd

BASE = Path.cwd().parent
ORIGINALES = BASE / "datos" / "originales"
LIMPIOS = BASE / "datos" / "limpios"
LIMPIOS.mkdir(exist_ok=True)

## 1. Carga

In [ ]:
df = pd.read_excel(ORIGINALES / "Base de datos afiliados ACTHO_anonimizada.xlsx", sheet_name="Hoja1")
assert df.shape == (472, 4), df.shape
df.head()

## 2. Renombrar columnas
Los nombres anonimizados (`Nombre de Empresa [anonimizado]`) en realidad contienen ids (`EMP-0002`, `P-02901`), así que se nombran como tales.

In [ ]:
df.columns = ["id_empresa", "ciudad", "nivel_cargo", "id_persona"]
for c in df.columns:
    df[c] = df[c].str.strip()
df.dtypes

## 3. Ciudad
Las ciudades ya vienen bien escritas, salvo 3 sin tilde: `Chia`, `Itagui` y `Sopo`. Se corrigen para que coincidan con las demás bases (`Chía`, `Itagüí`, `Sopó`).

In [ ]:
df["ciudad"] = df["ciudad"].replace({"Chia": "Chía", "Itagui": "Itagüí", "Sopo": "Sopó"})
df["ciudad"].value_counts().head(10)

## 4. Nivel de cargo
22 afiliados tienen `Sin dato`. Se deja así, en vez de cambiarlo a "No especificado", porque es la misma etiqueta que usan las demás bases para los vacíos.
Así en Looker Studio todos los filtros muestran una sola opción para "no sabemos".

In [ ]:
CARGOS = {"Dirección / Gerencia", "Alta dirección (C-level / VP)", "Jefatura / Coordinación",
          "Profesional / Analista / Consultor", "Estudiante / Practicante", "Otro", "Sin dato"}
assert set(df["nivel_cargo"]) <= CARGOS, set(df["nivel_cargo"]) - CARGOS
df["nivel_cargo"].value_counts()

## 5. Validaciones
- Sin vacíos.
- Cada empresa y cada afiliado aparece una sola vez (no hay duplicados).
- Los ids tienen el formato de las demás bases.

In [ ]:
assert df.notna().all().all()
assert df["id_empresa"].is_unique and df["id_persona"].is_unique
assert df["id_empresa"].str.fullmatch(r"EMP-\d{4}").all()
assert df["id_persona"].str.fullmatch(r"P-\d{5}").all()

df = df[["id_empresa", "id_persona", "nivel_cargo", "ciudad"]]
df.shape

## 6. Guardar
`utf-8-sig` para que Excel y Looker Studio lean bien las tildes.

In [ ]:
salida = LIMPIOS / "05_afiliados_actho_limpio.csv"
df.to_csv(salida, index=False, encoding="utf-8-sig")

assert pd.read_csv(salida).shape == (472, 4)
salida.name